# Notebook 12 — Gap Distribution Modeling

**prime-numbers-lab**

This notebook pivots from deterministic expected-gap modeling to **distributional gap modeling**.

Previous notebooks showed:

- `log(x)` is already a strong expected-gap baseline.
- affine / sinusoidal / rolling mean corrections barely improve RMSE.
- residual structure persists because gap variance dominates mean-bias correction.

Notebook 12 tests:

> Prime gaps should be modeled as a distribution around `log(x)`, not as a single deterministic curve.

Constraint → signal > noise

In [ ]:
from pathlib import Path
import math
import json
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

NOTEBOOK_ID = "12"
NOTEBOOK_SLUG = "gap_distribution_modeling"
RUN_NAME = f"{NOTEBOOK_ID}_{NOTEBOOK_SLUG}"

ROOT = Path(".")
DATA_DIR = ROOT / "data" / RUN_NAME
DOCS_DIR = ROOT / "docs" / RUN_NAME
FIG_DIR = ROOT / "figures" / RUN_NAME
TEX_DIR = ROOT / "tex" / RUN_NAME

for d in [DATA_DIR, DOCS_DIR, FIG_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Run:", RUN_NAME)

## 1. Helpers

We generate primes, compute consecutive prime gaps, and compare normalized gaps:

\[
z = \frac{g}{\log x}
\]

against an exponential baseline.

In [ ]:
def sieve_primes(n: int) -> np.ndarray:
    if n < 2:
        return np.array([], dtype=np.int64)
    sieve = np.ones(n + 1, dtype=bool)
    sieve[:2] = False
    limit = int(math.isqrt(n))
    for p in range(2, limit + 1):
        if sieve[p]:
            sieve[p*p:n+1:p] = False
    return np.flatnonzero(sieve).astype(np.int64)

def safe_kl(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float) + eps
    q = np.asarray(q, dtype=float) + eps
    p = p / p.sum()
    q = q / q.sum()
    return float(np.sum(p * np.log(p / q)))

def js_divergence(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float) + eps
    q = np.asarray(q, dtype=float) + eps
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)
    return 0.5 * safe_kl(p, m, eps=eps) + 0.5 * safe_kl(q, m, eps=eps)

def exp_pdf(z):
    z = np.asarray(z, dtype=float)
    return np.exp(-z)

def exp_cdf(z):
    z = np.asarray(z, dtype=float)
    return 1.0 - np.exp(-z)

def ecdf_values(samples):
    x = np.sort(np.asarray(samples, dtype=float))
    y = np.arange(1, len(x) + 1) / len(x)
    return x, y

def savefig(name):
    path = FIG_DIR / name
    plt.savefig(path, dpi=180, bbox_inches="tight")
    print("saved:", path)

## 2. Generate prime-gap data

In [ ]:
N_MAX = 1_000_000

primes = sieve_primes(N_MAX)
gaps = np.diff(primes)
x_vals = primes[:-1]
log_x = np.log(x_vals)
normalized_gap = gaps / log_x

gap_df = pd.DataFrame({
    "x": x_vals,
    "next_prime": primes[1:],
    "gap": gaps,
    "log_x": log_x,
    "normalized_gap": normalized_gap,
})

gap_df.to_csv(DATA_DIR / "12_prime_gap_data.csv", index=False)

print("prime count:", len(primes))
print("gap count:", len(gap_df))
gap_df.head()

## 3. Scale windows

In [ ]:
window_edges = np.unique(np.logspace(2, 6, 17).astype(int))
windows = list(zip(window_edges[:-1], window_edges[1:]))

def build_window_summary(df, windows):
    rows = []
    for lo, hi in windows:
        sub = df[(df["x"] >= lo) & (df["x"] < hi)]
        if len(sub) < 20:
            continue
        z = sub["normalized_gap"].to_numpy()
        g = sub["gap"].to_numpy()
        lx = sub["log_x"].to_numpy()
        rows.append({
            "x_lo": lo,
            "x_hi": hi,
            "x_mid": math.sqrt(lo * hi),
            "n": len(sub),
            "mean_gap": float(np.mean(g)),
            "mean_log_x": float(np.mean(lx)),
            "mean_z": float(np.mean(z)),
            "std_z": float(np.std(z)),
            "q50_z": float(np.quantile(z, 0.50)),
            "q75_z": float(np.quantile(z, 0.75)),
            "q90_z": float(np.quantile(z, 0.90)),
            "q95_z": float(np.quantile(z, 0.95)),
            "q99_z": float(np.quantile(z, 0.99)),
        })
    return pd.DataFrame(rows)

summary_df = build_window_summary(gap_df, windows)
summary_df.to_csv(DATA_DIR / "12_window_distribution_summary.csv", index=False)
summary_df

## 4. Normalized gap distribution vs exponential baseline

If \(z = g/\log(x)\), the classical random-prime heuristic suggests comparison with `Exp(1)`.

In [ ]:
z = gap_df["normalized_gap"].to_numpy()
z_plot = z[(z >= 0) & (z <= 8)]

plt.figure(figsize=(11, 6))
plt.hist(z_plot, bins=80, density=True, alpha=0.55, label="empirical normalized gaps")
grid = np.linspace(0, 8, 500)
plt.plot(grid, exp_pdf(grid), linewidth=2.5, label="Exponential(1) pdf")
plt.title("Normalized prime gaps versus exponential baseline")
plt.xlabel("normalized gap z = gap / log(x)")
plt.ylabel("density")
plt.legend()
plt.grid(True, alpha=0.35)
savefig("12_normalized_gap_exponential_overlay.png")
plt.show()

## 5. ECDF comparison

In [ ]:
x_ecdf, y_ecdf = ecdf_values(z_plot)
grid = np.linspace(0, np.percentile(z_plot, 99.5), 500)

plt.figure(figsize=(11, 6))
plt.plot(x_ecdf, y_ecdf, linewidth=2, label="empirical ECDF")
plt.plot(grid, exp_cdf(grid), linewidth=2.5, linestyle="--", label="Exponential(1) CDF")
plt.title("ECDF of normalized gaps")
plt.xlabel("normalized gap z")
plt.ylabel("cumulative probability")
plt.legend()
plt.grid(True, alpha=0.35)
savefig("12_normalized_gap_ecdf.png")
plt.show()

## 6. Windowed distribution distance

In [ ]:
bins = np.linspace(0, 8, 65)
dist_rows = []

for lo, hi in windows:
    sub = gap_df[(gap_df["x"] >= lo) & (gap_df["x"] < hi)]
    if len(sub) < 20:
        continue
    zs = sub["normalized_gap"].to_numpy()
    zs = zs[(zs >= 0) & (zs <= 8)]
    hist, _ = np.histogram(zs, bins=bins, density=False)
    p_emp = hist / max(hist.sum(), 1)

    p_exp = exp_cdf(bins[1:]) - exp_cdf(bins[:-1])
    p_exp = p_exp / p_exp.sum()

    dist_rows.append({
        "x_lo": lo,
        "x_hi": hi,
        "x_mid": math.sqrt(lo * hi),
        "n": len(sub),
        "kl_emp_exp": safe_kl(p_emp, p_exp),
        "js_emp_exp": js_divergence(p_emp, p_exp),
        "mean_z": float(np.mean(zs)),
        "std_z": float(np.std(zs)),
    })

dist_df = pd.DataFrame(dist_rows)
dist_df.to_csv(DATA_DIR / "12_distribution_distance_by_window.csv", index=False)

plt.figure(figsize=(11, 6))
plt.plot(dist_df["x_mid"], dist_df["js_emp_exp"], marker="o", label="JS divergence to Exp(1)")
plt.xscale("log")
plt.title("Windowed distribution distance to exponential baseline")
plt.xlabel("window midpoint x")
plt.ylabel("Jensen-Shannon divergence")
plt.legend()
plt.grid(True, alpha=0.35)
savefig("12_js_divergence_by_window.png")
plt.show()

dist_df

## 7. Quantile scaling

For an exponential model:

\[
Q_p(g \mid x) \approx -\log(1-p)\log(x)
\]

In [ ]:
quant_df = summary_df[["x_lo", "x_hi", "x_mid", "q50_z", "q75_z", "q90_z", "q95_z", "q99_z"]].copy()
quant_df.to_csv(DATA_DIR / "12_normalized_quantiles_by_window.csv", index=False)

plt.figure(figsize=(11, 6))
for col in ["q50_z", "q75_z", "q90_z", "q95_z", "q99_z"]:
    plt.plot(quant_df["x_mid"], quant_df[col], marker="o", label=col)

for p in [0.50, 0.75, 0.90, 0.95, 0.99]:
    plt.axhline(-math.log(1-p), linestyle="--", linewidth=1.1, alpha=0.55)

plt.xscale("log")
plt.title("Normalized gap quantiles by scale window")
plt.xlabel("window midpoint x")
plt.ylabel("quantile of gap / log(x)")
plt.legend(ncol=2)
plt.grid(True, alpha=0.35)
savefig("12_normalized_quantiles_by_window.png")
plt.show()

quant_df

## 8. Variance scaling

In [ ]:
var_rows = []
for lo, hi in windows:
    sub = gap_df[(gap_df["x"] >= lo) & (gap_df["x"] < hi)]
    if len(sub) < 20:
        continue
    mean_log = float(sub["log_x"].mean())
    var_rows.append({
        "x_lo": lo,
        "x_hi": hi,
        "x_mid": math.sqrt(lo * hi),
        "mean_gap": float(sub["gap"].mean()),
        "std_gap": float(sub["gap"].std()),
        "mean_log_x": mean_log,
        "mean_gap_over_log": float(sub["gap"].mean() / mean_log),
        "std_gap_over_log": float(sub["gap"].std() / mean_log),
        "variance_gap_over_log2": float(sub["gap"].var() / (mean_log ** 2)),
    })

var_df = pd.DataFrame(var_rows)
var_df.to_csv(DATA_DIR / "12_variance_scaling_by_window.csv", index=False)

plt.figure(figsize=(11, 6))
plt.plot(var_df["x_mid"], var_df["mean_gap_over_log"], marker="o", label="mean(gap) / mean(log x)")
plt.plot(var_df["x_mid"], var_df["std_gap_over_log"], marker="o", label="std(gap) / mean(log x)")
plt.axhline(1.0, linestyle="--", linewidth=1.5, label="Exp(1) reference")
plt.xscale("log")
plt.title("Mean and spread scaling relative to log(x)")
plt.xlabel("window midpoint x")
plt.ylabel("normalized value")
plt.legend()
plt.grid(True, alpha=0.35)
savefig("12_variance_scaling.png")
plt.show()

var_df

## 9. Tail exceedance rates

For \(z = g/\log x\), exponential baseline predicts:

\[
P(z > t) = e^{-t}
\]

In [ ]:
thresholds = [1, 2, 3, 4, 5]
tail_rows = []

for lo, hi in windows:
    sub = gap_df[(gap_df["x"] >= lo) & (gap_df["x"] < hi)]
    if len(sub) < 20:
        continue
    zlocal = sub["normalized_gap"].to_numpy()
    row = {"x_lo": lo, "x_hi": hi, "x_mid": math.sqrt(lo * hi), "n": len(sub)}
    for t in thresholds:
        row[f"emp_P_z_gt_{t}"] = float(np.mean(zlocal > t))
        row[f"exp_P_z_gt_{t}"] = float(math.exp(-t))
        row[f"tail_error_{t}"] = row[f"emp_P_z_gt_{t}"] - row[f"exp_P_z_gt_{t}"]
    tail_rows.append(row)

tail_df = pd.DataFrame(tail_rows)
tail_df.to_csv(DATA_DIR / "12_tail_exceedance_by_window.csv", index=False)

plt.figure(figsize=(11, 6))
for t in thresholds:
    plt.plot(tail_df["x_mid"], tail_df[f"emp_P_z_gt_{t}"], marker="o", label=f"emp P(z>{t})")
    plt.axhline(math.exp(-t), linestyle="--", linewidth=1.0, alpha=0.55)

plt.xscale("log")
plt.yscale("log")
plt.title("Tail exceedance rates for normalized gaps")
plt.xlabel("window midpoint x")
plt.ylabel("tail probability")
plt.legend(ncol=2)
plt.grid(True, alpha=0.35)
savefig("12_tail_exceedance_rates.png")
plt.show()

tail_df

## 10. Normalized gap heatmap

In [ ]:
heat_rows = []
heat_labels = []

for lo, hi in windows:
    sub = gap_df[(gap_df["x"] >= lo) & (gap_df["x"] < hi)]
    if len(sub) < 20:
        continue
    zs = sub["normalized_gap"].to_numpy()
    zs = zs[(zs >= 0) & (zs <= 8)]
    hist, _ = np.histogram(zs, bins=bins, density=True)
    heat_rows.append(hist)
    heat_labels.append(f"{lo}-{hi}")

heat = np.array(heat_rows)

plt.figure(figsize=(13, 7))
plt.imshow(heat, aspect="auto", origin="lower", extent=[bins[0], bins[-1], 0, len(heat_labels)])
plt.colorbar(label="density")
plt.yticks(np.arange(len(heat_labels)) + 0.5, heat_labels)
plt.title("Normalized gap distribution heatmap by x-window")
plt.xlabel("normalized gap z = gap / log(x)")
plt.ylabel("x window")
savefig("12_distribution_heatmap.png")
plt.show()

## 11. Distributional score

A compact diagnostic combining:

- JS divergence
- normalized mean error
- normalized standard deviation error
- tail exceedance error

In [ ]:
score_df = dist_df.merge(
    var_df[["x_mid", "mean_gap_over_log", "std_gap_over_log"]],
    on="x_mid",
    how="left"
).merge(
    tail_df[["x_mid"] + [f"tail_error_{t}" for t in thresholds]],
    on="x_mid",
    how="left"
)

tail_error_cols = [f"tail_error_{t}" for t in thresholds]
score_df["mean_abs_tail_error"] = score_df[tail_error_cols].abs().mean(axis=1)

score_df["distributional_score"] = 1 / (
    1
    + score_df["js_emp_exp"]
    + (score_df["mean_gap_over_log"] - 1).abs()
    + (score_df["std_gap_over_log"] - 1).abs()
    + score_df["mean_abs_tail_error"]
)

score_df.to_csv(DATA_DIR / "12_distributional_score_by_window.csv", index=False)

plt.figure(figsize=(11, 6))
plt.plot(score_df["x_mid"], score_df["distributional_score"], marker="o", label="distributional score")
plt.xscale("log")
plt.title("Distributional fit score by scale window")
plt.xlabel("window midpoint x")
plt.ylabel("score")
plt.legend()
plt.grid(True, alpha=0.35)
savefig("12_distributional_score.png")
plt.show()

score_df

## 12. Interpretation export

In [ ]:
interpretation = f"""# Gap Distribution Modeling (Notebook 12)

Notebook 12 pivots from deterministic expected-gap models to distributional gap modeling.

Previous notebooks showed that replacing `log(x)` with affine, sinusoidal, rolling, or hybrid expected-gap curves barely improves RMSE. This means gap error is dominated by distributional spread rather than smooth mean bias.

---

## 1. Normalized gap distribution

![Normalized gap exponential overlay](../../figures/{RUN_NAME}/12_normalized_gap_exponential_overlay.png)

We define:

```text
z = gap / log(x)
```

If prime gaps are modeled distributionally, `z` can be compared to an exponential baseline with mean 1.

---

## 2. ECDF comparison

![Normalized gap ECDF](../../figures/{RUN_NAME}/12_normalized_gap_ecdf.png)

The ECDF comparison avoids some histogram sensitivity.

Reference:

```text
F(z) = 1 - exp(-z)
```

---

## 3. Distribution distance by window

![JS divergence by window](../../figures/{RUN_NAME}/12_js_divergence_by_window.png)

Each scale window produces an empirical normalized-gap distribution.

We compare each window to the exponential baseline using Jensen-Shannon divergence.

---

## 4. Quantile scaling

![Normalized quantiles by window](../../figures/{RUN_NAME}/12_normalized_quantiles_by_window.png)

Exponential reference quantiles:

```text
q50 ≈ 0.693
q90 ≈ 2.303
q95 ≈ 2.996
q99 ≈ 4.605
```

---

## 5. Variance scaling

![Variance scaling](../../figures/{RUN_NAME}/12_variance_scaling.png)

For an exponential model with mean `log(x)`, both mean and standard deviation scale like `log(x)`.

---

## 6. Tail exceedance rates

![Tail exceedance rates](../../figures/{RUN_NAME}/12_tail_exceedance_rates.png)

For normalized gap `z`:

```text
P(z > t) = exp(-t)
```

---

## 7. Distribution heatmap

![Distribution heatmap](../../figures/{RUN_NAME}/12_distribution_heatmap.png)

Rows are x-windows. Columns are normalized gap bins.

---

## 8. Distributional score

![Distributional score](../../figures/{RUN_NAME}/12_distributional_score.png)

The score combines distribution distance, normalized mean/spread error, and tail exceedance error.

---

## Core result

Notebook 12 supports the transition:

```text
expected gap model → gap distribution model
```

In other words:

```text
gap ≠ only f(x)
gap ~ distribution around log(x)
```

Constraint → signal > noise
"""

(DOCS_DIR / "interpretation.md").write_text(interpretation)
print("wrote:", DOCS_DIR / "interpretation.md")

In [ ]:
tex = """\\section*{Notebook 12: Gap Distribution Modeling}

Notebook 12 reframes prime-gap reconstruction as distribution modeling.

Let \\(g(x)\\) denote the next-prime gap after \\(x\\). Earlier deterministic models tested
\\[
g(x) \\approx f(x)
\\]
with \\(f(x)\\) based on \\(\\log x\\), affine corrections, sinusoidal corrections, and local rolling averages.
Those models produced negligible improvement over \\(\\log x\\).

Notebook 12 instead studies
\\[
z = \\frac{g(x)}{\\log x}.
\\]
The exponential heuristic suggests
\\[
z \\sim \\mathrm{Exp}(1),
\\]
or equivalently
\\[
P(z > t) \\approx e^{-t}.
\\]

The notebook evaluates empirical histograms, ECDFs, quantiles, variance scaling, and windowed
distribution distances. The central conclusion is that gap variability is better evaluated as
a distribution around \\(\\log x\\) than as a deterministic correction to \\(\\log x\\).

\\[
\\text{Constraint} \\rightarrow \\text{signal} > \\text{noise}.
\\]
"""
(TEX_DIR / "12_gap_distribution_modeling.tex").write_text(tex)
print("wrote:", TEX_DIR / "12_gap_distribution_modeling.tex")

## 13. Export package

In [ ]:
EXPORT_NAME = "12_gap_distribution_modeling_export.zip"

with zipfile.ZipFile(EXPORT_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [DOCS_DIR, DATA_DIR, FIG_DIR, TEX_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"Export ready: {EXPORT_NAME}")
print("Tip: uncomment Colab lines below to download.")

# --- Optional Colab download ---
# Uncomment the lines below when running in Colab
#
# from google.colab import files
# files.download(EXPORT_NAME)